# A2 欧氏 AE 与 A3 类别平衡 AE

独立 kernel、单卡顺序运行两组；默认 3 个切片 × 5 个 seed。完整协议见 [README](README.md#a0a5-欧氏对照实验)。本轮是探索性表示实验，不使用隐藏切片选参。


In [1]:
# 必须在导入 torch 前运行；切换显卡或更新代码后重启 kernel。
import os
import sys
GPU_ID = "5"  # 调度器管理可见卡时设为 None。
if "torch" in sys.modules:
    raise RuntimeError("请重启独立 kernel，并从第一格运行。")
if GPU_ID is not None:
    os.environ["CUDA_VISIBLE_DEVICES"] = GPU_ID
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
print("CUDA_VISIBLE_DEVICES =", os.environ.get("CUDA_VISIBLE_DEVICES", "scheduler/default"))


CUDA_VISIBLE_DEVICES = 5


In [2]:
from pathlib import Path
from copy import deepcopy
PROJECT = next((candidate for parent in (Path.cwd(), *Path.cwd().parents)
                for candidate in (parent, parent / "MMDeepSpatial")
                if (candidate / "mmdeepspatial/euclidean.py").is_file()), None)
if PROJECT is None:
    raise FileNotFoundError("请在 MMDeepSpatial 或其父目录打开 notebook。")
sys.path.insert(0, str(PROJECT))
import pandas as pd
from mmdeepspatial.euclidean import EUCLIDEAN_CONFIG, run_arm, compare_arms
from mmdeepspatial.data import input_paths
print("Project:", PROJECT)


Project: /3250608056/code/DeepSpatial/MMDeepSpatial


## 统一配置

三个 notebook 的科学配置须一致。A3/A5 使用训练标签计算重建损失权重，probe 设置不变。GCN 只用可见同切片表达构图，新细胞仅连接冻结训练参考库，不使用隐藏 XY 或查询之间的边。

若做短流程检查，统一改成一个 holdout、一个 seed、两轮训练，并将 `REUSE_A2_ROOT=None`，另用测试输出目录。正式运行恢复全部配置。


In [3]:
ARMS_TO_RUN = ['A2', 'A3']
DATA_GLOB = str(PROJECT.parent / "data/merfish_mouse_hypothalamus/**/merfish_[0-4].h5ad")
OUTPUT_ROOT = PROJECT / "outputs/euclidean_v1"
HOLDOUTS = [1, 2, 3]
SEEDS = [42, 0, 1, 2, 3]
EPOCHS = 100
DIM = 32
HIDDEN = 128
GRAPH_K = 10
BALANCE_POWER = 0.5  # 逆平方根频率，样本权重全训练集均值为 1。
LABEL_KEY = "cell_class"
LAYER = None
TRANSFORM = "log1p"  # X 已 log / 标准化时统一改 identity。
DEVICE = "cuda"
# 可复用上一轮兼容的 E0 AE；设为 None 则重新训练 A2。
REUSE_A2_ROOT = PROJECT / "outputs/representations_v2/E0_ae"
if not REUSE_A2_ROOT.is_dir():
    REUSE_A2_ROOT = None
CONFIG = deepcopy(EUCLIDEAN_CONFIG)
CONFIG.update(input_glob=DATA_GLOB, output_root=str(OUTPUT_ROOT), holdouts=HOLDOUTS,
              seeds=SEEDS, epochs=EPOCHS, dim=DIM, hidden=HIDDEN, graph_k=GRAPH_K,
              balance_power=BALANCE_POWER, label_key=LABEL_KEY, layer=LAYER,
              transform=TRANSFORM, device=DEVICE,
              reuse_a2_root=str(REUSE_A2_ROOT) if REUSE_A2_ROOT else None)
CONFIG


{'input_glob': '/3250608056/code/DeepSpatial/data/merfish_mouse_hypothalamus/**/merfish_[0-4].h5ad',
 'output_root': '/3250608056/code/DeepSpatial/MMDeepSpatial/outputs/euclidean_v1',
 'holdouts': [1, 2, 3],
 'seeds': [42, 0, 1, 2, 3],
 'label_key': 'cell_class',
 'layer': None,
 'transform': 'log1p',
 'n_genes': 0,
 'rare_fraction': 0.05,
 'dim': 32,
 'hidden': 128,
 'epochs': 100,
 'batch_size': 512,
 'lr': 0.001,
 'weight_decay': 1e-05,
 'grad_clip': 5.0,
 'curvature': 1.0,
 'fusion_alpha': 0.5,
 'log_every': 10,
 'e0_method': 'ae',
 'device': 'cuda',
 'cpu_threads': 4,
 'retrieval_ks': [1, 5, 15],
 'query_chunk': 128,
 'reference_chunk': 2048,
 'coverage_quantile': 0.95,
 'coverage_calibration_max': 2048,
 'probe_c': 1.0,
 'probe_class_weight': 'balanced',
 'probe_max_iter': 2000,
 'shuffled_probe': False,
 'max_tangent_radius': 1.5,
 'diagnostics': False,
 'diagnostic_alphas': [0.25, 0.75],
 'graph_k': 10,
 'graph_query_chunk': 512,
 'balance_power': 0.5,
 'reuse_a2_root': '/32506

In [4]:
paths = input_paths(CONFIG["input_glob"])
display(pd.DataFrame({"slice_index": range(len(paths)), "file": [str(p) for p in paths],
                      "held_out": [i in HOLDOUTS for i in range(len(paths))]}))
print("Arms:", ARMS_TO_RUN, "runs per arm:", len(HOLDOUTS)*len(SEEDS))
print("A0/A1 不训练神经编码器；A2 可复用已核验的旧权重。")


,slice_index,file,held_out
0,0,/3250608056/code/DeepSpatial/data/merfish_mous...,False
1,1,/3250608056/code/DeepSpatial/data/merfish_mous...,True
2,2,/3250608056/code/DeepSpatial/data/merfish_mous...,True
3,3,/3250608056/code/DeepSpatial/data/merfish_mous...,True
4,4,/3250608056/code/DeepSpatial/data/merfish_mous...,False


Arms: ['A2', 'A3'] runs per arm: 15
A0/A1 不训练神经编码器；A2 可复用已核验的旧权重。


In [5]:
results = pd.concat([run_arm(arm, CONFIG) for arm in ARMS_TO_RUN], ignore_index=True)
display(results[["arm", "holdout", "seed", "encoder_uses_labels", "encoder_reused",
                 "rare_macro_auprc", "ultra_rare_macro_auprc", "low_rare_macro_auprc",
                 "rare_macro_f1", "rare_macro_retrieval_hit_at_5", "common_macro_f1",
                 "trainable_parameters"]])


Start A2 holdout=1 seed=42 train=(22760, 155) method=ae balanced=False
 rare_macro_auprc  ultra_rare_macro_auprc  rare_macro_f1  overall_accuracy
         0.700684                0.483898       0.631139          0.901925
Start A2 holdout=1 seed=0 train=(22760, 155) method=ae balanced=False
 rare_macro_auprc  ultra_rare_macro_auprc  rare_macro_f1  overall_accuracy
         0.686604                0.459635        0.60884          0.899586
Start A2 holdout=1 seed=1 train=(22760, 155) method=ae balanced=False
 rare_macro_auprc  ultra_rare_macro_auprc  rare_macro_f1  overall_accuracy
         0.674343                0.431616       0.623312          0.886629
Start A2 holdout=1 seed=2 train=(22760, 155) method=ae balanced=False
 rare_macro_auprc  ultra_rare_macro_auprc  rare_macro_f1  overall_accuracy
          0.73242                0.545404        0.64944          0.897067
Start A2 holdout=1 seed=3 train=(22760, 155) method=ae balanced=False
 rare_macro_auprc  ultra_rare_macro_auprc  rare_m

,arm,holdout,seed,encoder_uses_labels,encoder_reused,rare_macro_auprc,ultra_rare_macro_auprc,low_rare_macro_auprc,rare_macro_f1,rare_macro_retrieval_hit_at_5,common_macro_f1,trainable_parameters
0,A2,1,42,False,True,0.700684,0.483898,0.960827,0.631139,0.700377,0.922246,48315
1,A2,1,0,False,True,0.686604,0.459635,0.958967,0.608840,0.731679,0.921013,48315
2,A2,1,1,False,True,0.674343,0.431616,0.965615,0.623312,0.757543,0.913832,48315
3,A2,1,2,False,True,0.732420,0.545404,0.956839,0.649440,0.734777,0.915014,48315
4,A2,1,3,False,True,0.678349,0.442463,0.961412,0.627611,0.741499,0.918636,48315
5,A2,2,42,False,True,0.771321,0.623144,0.949134,0.600488,0.718445,0.905185,48315
6,A2,2,0,False,True,0.783237,0.643453,0.950978,0.632495,0.747680,0.909742,48315
7,A2,2,1,False,True,0.810298,0.688202,0.956814,0.629721,0.755852,0.909228,48315
8,A2,2,2,False,True,0.768182,0.612052,0.955539,0.608682,0.755829,0.908222,48315
9,A2,2,3,False,True,0.799476,0.671929,0.952533,0.623556,0.752943,0.915212,48315


In [6]:
metrics = ["rare_macro_auprc", "ultra_rare_macro_auprc", "low_rare_macro_auprc",
           "rare_macro_f1", "rare_macro_recall", "rare_macro_retrieval_hit_at_5",
           "rare_macro_knn_purity_at_15", "common_macro_f1"]
by_fold = results.groupby(["arm", "holdout"])[metrics].mean()
display(by_fold)
# 描述性切片间统计，不是独立生物学重复的置信区间。
display(by_fold.groupby("arm").agg(["mean", "std"]))


rare_macro_auprc  ultra_rare_macro_auprc  low_rare_macro_auprc  \
arm holdout                                                                   
A2  1                0.694480                0.472603              0.960732   
    2                0.786503                0.647756              0.952999   
    3                0.726029                0.532111              0.958731   
A3  1                0.763589                0.592827              0.968504   
    2                0.811973                0.684913              0.964446   
    3                0.751059                0.572827              0.964936   

             rare_macro_f1  rare_macro_recall  rare_macro_retrieval_hit_at_5  \
arm holdout                                                                    
A2  1             0.628068           0.750150                       0.733175   
    2             0.618988           0.860996                       0.746150   
    3             0.636953           0.802890                       0.772066   
A3  1             0.691277           0.798584                       0.760719   
    2             0.674334           0.850063                       0.794522   
    3             0.665681           0.790974                       0.783851   

             rare_macro_knn_purity_at_15  common_macro_f1  
arm holdout                                                
A2  1                           0.527808         0.918148  
    2                           0.554355         0.909518  
    3                           0.574752         0.922748  
A3  1                           0.545525         0.926586  
    2                           0.564013         0.925703  
    3                           0.570769         0.930366

rare_macro_auprc           ultra_rare_macro_auprc            \
                mean       std                   mean       std   
arm                                                               
A2          0.735671  0.046763               0.550823  0.089063   
A3          0.775540  0.032168               0.616856  0.059782   

    low_rare_macro_auprc           rare_macro_f1           rare_macro_recall  \
                    mean       std          mean       std              mean   
arm                                                                            
A2              0.957487  0.004013      0.628003  0.008982          0.804679   
A3              0.965962  0.002215      0.677097  0.013020          0.813207   

              rare_macro_retrieval_hit_at_5            \
          std                          mean       std   
arm                                                     
A2   0.055445                      0.750464  0.019801   
A3   0.032144                      0.779698  0.017280   

    rare_macro_knn_purity_at_15           common_macro_f1            
                           mean       std            mean       std  
arm                                                                  
A2                     0.552305  0.023539        0.916805  0.006717  
A3                     0.560102  0.013069        0.927552  0.002477

## 六组全部完成后统一比较

任何一个 notebook 将下方开关改为 True 后运行。比较会核对完整 seed/split、输入、预处理、源码、容量和 A4/A5 图一致性。保存六组原始指标、稀有度分层、逐类表及预定义配对差值，不自动选择最佳模型。


In [7]:
COMPARE_WHEN_READY = False
if COMPARE_WHEN_READY:
    all_runs, aggregate, all_types = compare_arms(OUTPUT_ROOT)
    display(aggregate)
    display(pd.read_csv(OUTPUT_ROOT / "comparison/paired_deltas.csv"))
